#**An Exploratory Data Analysis (EDA) for the FMA-Small Dataset**
**Submitted by: Elia Fel P. Bado**

```markdown
# 1. Introduction

Data Source & Origin
Repository: [FMA GitHub Repository](https://github.com/mdeff/fma)
Citation: Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. arXiv:1612.01840.

### Context & Purpose
* **Who collected the data, when, how, and for what purpose?**
  The dataset was created by Michaël Defferrard, Kirell Benzi, Pierre Vandergheynst, and Xavier Bresson in 2017 for the 18th International Society for Music Information Retrieval Conference (ISMIR). It was designed to provide an open, easily accessible audio dataset for evaluating tasks in Music Information Retrieval (MIR), aiming to overcome hurdles from the limited availability of large public audio datasets. The data was collected by querying the Free Music Archive (FMA) web API.

### Data Granularity
* **What does one row represent?**
  The representation depends on the metadata table:
  * `tracks.csv`: One row represents a single music track.
  * `genres.csv`: One row represents a single musical genre.
  * `features.csv`: One row represents extracted audio features (via `librosa`) for a track.
  * `echonest.csv`: One row represents Echonest-specific audio features for a track.

### Data Dictionary Overview
* **What does each column mean?**
  * **`tracks.csv`**: Contains track metadata such as track ID, title, artist, genres, tags, and play counts.
  * **`genres.csv`**: Details 163 genres, including name, ID, and parent genre ID to infer hierarchy.
  * **`features.csv`**: Contains statistical descriptors of common audio features extracted using `librosa` (e.g., MFCCs, spectral centroid, chroma).
  * **`echonest.csv`**: Provides Echonest audio features for a subset of 13,129 tracks.
  * **Audio Files**: Raw audio is provided in MP3 format (30-second clips for the *FMA-Small* subset, or untrimmed tracks for others).

### Questions to Explore
1. **Genre Recognition**: How accurately can baseline machine learning models perform genre recognition using extracted audio features vs. raw audio?
2. **Hierarchical Taxonomy**: How does the hierarchical genre taxonomy affect classification performance across top-level genres?
3. **Representation Learning**: Can representation learning on artist labels transfer effectively to genre classification tasks?
4. **Feature Comparison**: How do Echonest audio features compare with librosa-extracted features in predicting track play counts?
```

# 2. Data loading and first look

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# connecting colab to gdrive, check if folder exists, and return folder size
from google.colab import drive
drive.mount('/content/drive')

#paths
base_path = '/content/drive/MyDrive/Music_Genre_Classification'
dataset_path = os.path.join(base_path, 'dataset')

raw_dataset_path = os.path.join(dataset_path, 'raw')
fma_metadata_path = os.path.join(raw_dataset_path, 'fma_metadata')
fma_small_path = os.path.join(raw_dataset_path, 'fma_small')

processed_dataset_path = os.path.join(dataset_path, 'processed')


#for fetching folder size
def get_dir_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

#to check if folders has contents
folder_exists = os.path.exists(base_path)
print(f"Music_Genre_Classification file exists: {folder_exists}")
print(f"dataset/raw/fma_metadata exists: {os.path.exists(fma_metadata_path)}")
print(f"dataset/raw/fma_small exists: {os.path.exists(fma_small_path)}\n")

if folder_exists:
    fma_metadata_file_size = (get_dir_size(fma_metadata_path))/(1024*1024)
    fma_small_file_size = (get_dir_size(fma_small_path))/(1024*1024)
    print(f"fma_metadata Folder Size: {fma_metadata_file_size:.2f} MB")
    print(f"fma_small Folder Size: {fma_small_file_size:.2f} MB")
else:
    print("Folder Size: N/A")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Music_Genre_Classification file exists: True
dataset/raw/fma_metadata exists: True
dataset/raw/fma_small exists: True

fma_metadata Folder Size: 1396.98 MB
fma_small Folder Size: 7606.00 MB


In [14]:
tracks_path = os.path.join(fma_metadata_path, 'tracks.csv')

# load the raw metadata handling the multi-index header
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])

#print the shape, head, and info
print("--- Shape ---")
print(tracks.shape)
print("\n--- Head ---")
display(tracks.head())
print("\n--- Info ---")
tracks.info()

print("\n" + "="*50 + "\n")
print("Summary:\nThe raw tracks dataframe has the printed shape, which likely shows over 100,000 rows and multiple columns (with a multi-level index). This extensive raw data typically requires immediate filtering to focus only on the subset relevant to the FMA-Small dataset.")

--- Shape ---
(106574, 52)

--- Head ---


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]


--- Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 106574 entries, 2 to 155320
Data columns (total 52 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   (album, comments)            106574 non-null  int64  
 1   (album, date_created)        103045 non-null  object 
 2   (album, date_released)       70294 non-null   object 
 3   (album, engineer)            15295 non-null   object 
 4   (album, favorites)           106574 non-null  int64  
 5   (album, id)                  106574 non-null  int64  
 6   (album, information)         83149 non-null   object 
 7   (album, listens)             106574 non-null  int64  
 8   (album, producer)            18060 non-null   object 
 9   (album, tags)                106574 non-null  object 
 10  (album, title)               105549 non-null  object 
 11  (album, tracks)              106574 non-null  int64  
 12  (album, type)                100066 non-null  obj

In [13]:
# Filter tracks for the 'small' subset
tracks_small = tracks[tracks[('set', 'subset')] == 'small']

print("--- FMA-Small Metadata Shape ---")
print(tracks_small.shape)

# Let's verify the distribution of the top-level genres in this subset
print("--- Genre Distribution in FMA-Small ---")
genre_counts = tracks_small[('track', 'genre_top')].value_counts()
display(genre_counts)

# Quick validation that we have exactly 8,000 tracks (each with 1,000 tracks per 8 genres)
print(f"Total Tracks in FMA-Small: {len(tracks_small)}")

--- FMA-Small Metadata Shape ---
(8000, 52)
--- Genre Distribution in FMA-Small ---


,count
"(track, genre_top)",
Hip-Hop,1000
Pop,1000
Folk,1000
Experimental,1000
Rock,1000
International,1000
Electronic,1000
Instrumental,1000


Total Tracks in FMA-Small: 8000


```markdown
### Variable Types, Summaries, and Valid Plots

This table categorizes the columns in the FMA-Small dataset by their variable type, which dictates the appropriate statistical summaries and visualizations during Exploratory Data Analysis (EDA).

| Variable Type | Examples (Columns) | Summaries | Plots |
| :--- | :--- | :--- | :--- |
| **Quantitative (Continuous)** | `track.duration`, `track.listens`, `track.interest`, `album.listens` | Mean, Median, Standard Deviation, Min, Max, Skewness | Histogram, Density Plot (KDE), Box Plot, Violin Plot, Scatter Plot |
| **Quantitative (Discrete)** | `track.bit_rate`, `track.comments`, `track.favorites`, `album.tracks` | Mean, Median, Min, Max, Quantiles | Histogram, Box Plot, Bar Chart (if few unique values) |
| **Categorical (Nominal)** | `track.genre_top` (Target), `track.language_code`, `set.split` | Mode, Frequency Counts, Proportions / Percentages | Bar Chart, Count Plot, Pie Chart |
| **Categorical (Multi-label)** | `track.genres`, `track.genres_all`, `track.tags` | Mode, List Length, Co-occurrence counts | Bar Chart (Top N), Network Graph |
| **Text / Unstructured** | `track.title`, `artist.name`, `artist.bio` | Unique Count, Word Frequency | Word Cloud, Frequency Bar Chart |
| **Temporal / Date** | `track.date_created`, `album.date_released` | Min (Earliest), Max (Latest), Range | Line Chart (Time Series), Histogram (by year/month) |
| **Geospatial** | `artist.latitude`, `artist.longitude` | Min, Max, Mean (Centroid) | Scatter Plot, Map Overlay / Heatmap |
| **Identifier** | `track_id` (Index), `album.id`, `artist.id` | Unique Count, Missing Values | N/A |
```

# 3. Data quality assessment

In [16]:
# 1. Missing values: count and percent per column
missing = tracks_small.isna().sum().to_frame("n_missing")
missing["pct"] = (missing["n_missing"] / len(tracks_small) * 100).round(1)
missing = missing[missing["n_missing"] > 0].sort_values("pct", ascending=False)
print("--- Top 10 Columns with Missing Values ---")
display(missing.head(10))

# Pattern of missingness: Compare rows with and without missing values on another column
# Example: Does missingness in 'language_code' depend on the 'genre_top'?
col_with_missing = ('track', 'language_code')
target_col = ('track', 'genre_top')

if col_with_missing in tracks_small.columns and target_col in tracks_small.columns:
    print(f"\n--- Missingness Pattern: {col_with_missing[1]} vs {target_col[1]} ---")
    missing_mask = tracks_small[col_with_missing].isna()

    print("Genre distribution when language_code is MISSING (%):")
    display(tracks_small[missing_mask][target_col].value_counts(normalize=True).round(3) * 100)

    print("\nGenre distribution when language_code is NOT MISSING (%):")
    display(tracks_small[~missing_mask][target_col].value_counts(normalize=True).round(3) * 100)

# 2. Duplicate rows and duplicate IDs
n_duplicates = tracks_small.duplicated().sum()
n_dup_ids = tracks_small.index.duplicated().sum()  # track_id is the index
print(f"\n--- Duplicates ---")
print(f"Duplicate rows: {n_duplicates}")
print(f"Duplicate track IDs: {n_dup_ids}")

# 3. Outliers using the IQR rule (Example: track duration)
print("\n--- Outliers (IQR Rule) for Track Duration ---")
durations = tracks_small[('track', 'duration')]
Q1 = durations.quantile(0.25)
Q3 = durations.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tracks_small[(durations < lower_bound) | (durations > upper_bound)]
print(f"Number of outliers flagged in duration: {len(outliers)}")
print("Sample of outlier rows (Decide if error or extreme value):")
display(outliers[[('track', 'duration'), ('track', 'genre_top'), ('artist', 'name')]].head())

# 4. Impossible values: check against what is physically/logically possible
print("\n--- Impossible Values ---")
impossible_duration = tracks_small[tracks_small[('track', 'duration')] <= 0]
print(f"Tracks with duration <= 0 seconds: {len(impossible_duration)}")

--- Top 10 Columns with Missing Values ---


n_missing   pct
track  lyricist                7974  99.7
       publisher               7924  99.1
       information             7841  98.0
       composer                7820  97.8
artist active_year_end         7642  95.5
       wikipedia_page          7590  94.9
track  date_recorded           7535  94.2
artist related_projects        7306  91.3
       associated_labels       7151  89.4
track  language_code           6995  87.4


--- Missingness Pattern: language_code vs genre_top ---
Genre distribution when language_code is MISSING (%):


,proportion
"(track, genre_top)",
Instrumental,13.8
Electronic,13.7
Experimental,12.6
Pop,12.5
Hip-Hop,12.2
Rock,11.9
International,11.9
Folk,11.4



Genre distribution when language_code is NOT MISSING (%):


,proportion
"(track, genre_top)",
Folk,19.9
International,16.8
Rock,16.4
Hip-Hop,14.6
Pop,12.6
Experimental,11.5
Electronic,4.4
Instrumental,3.7



--- Duplicates ---
Duplicate rows: 0
Duplicate track IDs: 0

--- Outliers (IQR Rule) for Track Duration ---
Number of outliers flagged in duration: 270
Sample of outlier rows (Decide if error or extreme value):


track                       artist
         duration     genre_top           name
track_id                                      
615           519  Experimental            EKG
892           480          Folk    Ilyas Ahmed
997           520  Experimental  Kevin Shields
998           475  Experimental  Kevin Shields
1083          472  Experimental  Magic Lantern


--- Impossible Values ---
Tracks with duration <= 0 seconds: 0


# 4. Data cleaning
*(with a log of every change)*

# 5. Univariate analysis

# 6. Bivariate and multivariate analysis

# 7. Type-specific analysis
*(time, text, images, groups, as applies)*

# 8. Summary of findings

# 9. Limitations and next steps

# 10. References